https://www.youtube.com/watch?v=CXXl9CFY5ZU


In [ ]:
import os
import ee
import geemap
from dotenv import load_dotenv

from tools.Input_values.satellites import sentinel2_harmonized
from tools.Input_values.locaties import select_naturenreserves, build_location_bound
from tools.data_cleansing.sentinel2 import add_local_cloud_cover, scl_cloud_removal_20m, scale_reflectance
from tools.calculate_spectral_indices.sentinel2 import calc_ndvi, calc_evi


# Authenticate only once if needed
# ee.Authenticate(auth_mode='localhost', force=True)

load_dotenv()

project_code_ee = os.getenv("PROJECT_CODE_EE")

if not project_code_ee:
    raise RuntimeError("PROJECT_CODE_EE not found")

ee.Initialize(project=project_code_ee)

# Maak kaart
Map = geemap.Map()
Map.add_basemap("satellite")

# -----------------------------
# 1. Natuurgebied selecteren
# -----------------------------
natuurgebieden = select_naturenreserves(["Nieuwkoopse Plassen & de Haeck"])

# Controleer het gebied op de kaart
Map.centerObject(natuurgebieden, 11)
Map.addLayer(natuurgebieden, {"color": "red"}, "Natuurgebied")

# -----------------------------
# 4. Sentinel-2 collectie maken
# -----------------------------
s2 = (
    ee.ImageCollection(sentinel2_harmonized)
    .filterDate("2020-07-10", "2020-07-12")
    .filterBounds(natuurgebieden)
    .map(add_local_cloud_cover(natuurgebieden, coverage = "bounds"))
    .filter(ee.Filter.lte("LOCAL_CLOUD_COVER", 30))
    # .map(probability_cloud_removal_20m(35))
    .map(scl_cloud_removal_20m(remove_unclassified=False, bloat_n_meters= 50))
    .map(scale_reflectance)
    .map(calc_ndvi)
    .map(calc_evi)
)

# -----------------------------
# 5. Maak 1 NDVI eindbeeld
#    Hier nemen we de mediaan
#    van alle NDVI-beelden in de tijd
# -----------------------------
ndvi_gebied = s2.select("NDVI").median().clip(build_location_bound(locations_of_interest=natuurgebieden, mode="geometry"))
evi_gebied = s2.select("EVI").median().clip(build_location_bound(locations_of_interest=natuurgebieden, mode="geometry"))
rgb = s2.median().clip(build_location_bound(locations_of_interest=natuurgebieden, mode="geometry"))

# -----------------------------
# 6. Visualisatie
# -----------------------------
visTrueColor = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
    # 'gamma': 0.8,
}

visNDVI = {
    'bands': ['NDVI'],
    'min': -1,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}

visEVI = {
    'bands': ['EVI'],
    'min': -1,
    'max': 2,
    'palette': ['red', 'yellow', 'green']
}

Map.addLayer(ndvi_gebied, visNDVI, "NDVI natuurgebied")
Map.addLayer(evi_gebied, visEVI, "EVI natuurgebied")
Map.addLayer(rgb, visTrueColor, "RGB median")

# median() → robust to outliers (very popular for vegetation)
# sum() → total over time
# min() → lowest value per pixel
# max() → highest value per pixel
# mode() → most frequent value (useful for categorical data)
# stdDev() → variability over time
# variance() → statistical spread
# reduce(ee.Reducer...) → fully custom reduction

# clip(geometry) → hard crop to boundary
# clipToCollection(fc) → clip to multiple features
# updateMask(mask) → soft masking (very important alternative to clip)
# selfMask() → mask zeros automatically

Map

In [ ]:
import pandas as pd
from IPython.display import display
from tools.data_analysis.geemap_data import (
    describe_image_properties,
    image_collection_to_metadata_table,
)

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(describe_image_properties(s2, "sentinel2_sr_harmonized"))

In [ ]:
image_metadata = image_collection_to_metadata_table(
    collection=s2,
    locations_of_interest=natuurgebieden,
    coverage="bounds",
    buffer_meters=0,
    bands=["NDVI"],
    include_properties=[
        "CLOUDY_PIXEL_PERCENTAGE",
        "LOCAL_CLOUD_COVER",
        "MGRS_TILE",
    ],
    scale=10,
)
image_metadata

In [ ]:
from tools.extract_downloads.geemap_image import download_single_geotiff

download_single_geotiff(
    image=s2.first(),
    filename="output/ndvi_first_image.tif",
    locations_of_interest=natuurgebieden,
    coverage="buffer",
    buffer_meters= 100,
    bands=["NDVI"],
    scale=10,
)

In [ ]:
from tools.extract_downloads.geemap_image import convert_geotiff_to_visual_image

visNDVI = {
    'bands': ['NDVI'],
    'min': -1,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}

convert_geotiff_to_visual_image(
    tif_filename=r"output\ndvi_first_image.tif",
    output_filename=r"output\ndvi_first_image.png",
    vis_params= visNDVI

)